Koden läser in ett dataset `salary_dataset.csv` med löner och arbetslivserfarenhet och delar upp informationen i ett träningsset och ett testset. Därefter jämförs två olika maskininlärningsmodeller, linjär regression och random forest, genom korsvalidering på träningsdatan för att se vilken av dem som generellt gör minst fel. Slutligen tränas den modell som presterade bäst på hela träningssetet och utvärderas på det undanlagda testsetet för att ge ett slutgiltigt mått på hur väl den kan förutsäga löner för helt ny och osedd data.

In [2]:
# Importera nödvändiga bibliotek
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# Läs in datasetet med pandas
df = pd.read_csv('salary_dataset.csv')

# Välj ut X (oberoende variabel) och y (beroende variabel)
# Vi använder dubbla hakparanteser för X så att den förblir en DataFrame (2D-struktur)
# vilket scikit-learn kräver. y kan vara en vanlig Pandas Series (1D).
X = df[['YearsExperience']]
y = df['Salary']

print(f"Datasetet inläst! X har formen {X.shape} och y har formen {y.shape}.")

Datasetet inläst! X har formen (30, 1) och y har formen (30,).


In [4]:
# Dela upp datan: 80% för träning och 20% för test
# Vi använder random_state för att resultatet ska bli exakt likadant varje gång vi kör koden
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Träningssetet innehåller {len(X_train)} rader.")
print(f"Testsetet innehåller {len(X_test)} rader.")

Träningssetet innehåller 24 rader.
Testsetet innehåller 6 rader.


In [5]:
# Instantiera de två modellerna
lin_reg = LinearRegression()
rf_reg = RandomForestRegressor(random_state=42)

# Genomför 5-delad korsvalidering för Linjär Regression
cv_lin_reg = cross_validate(lin_reg, X_train, y_train, 
                            cv=5, 
                            scoring='neg_root_mean_squared_error')

# Genomför 5-delad korsvalidering för Random Forest
cv_rf_reg = cross_validate(rf_reg, X_train, y_train, 
                           cv=5, 
                           scoring='neg_root_mean_squared_error')

# Eftersom vi använde 'neg_root_mean_squared_error' blir poängen negativ.
# Vi multiplicerar med -1 och räknar ut medelvärdet av RMSE över alla 5 iterationer.
rmse_lin_reg = -np.mean(cv_lin_reg['test_score'])
rmse_rf_reg = -np.mean(cv_rf_reg['test_score'])

print(f"Korsvalidering RMSE för Linjär Regression: {rmse_lin_reg:.2f}")
print(f"Korsvalidering RMSE för Random Forest: {rmse_rf_reg:.2f}")

Korsvalidering RMSE för Linjär Regression: 5293.20
Korsvalidering RMSE för Random Forest: 4921.14


In [6]:
# Träna vinnarmodellen (Linjär Regression) på hela träningssetet
lin_reg.fit(X_train, y_train)

# Gör prediktioner på det undanlagda testsetet
y_pred = lin_reg.predict(X_test)

# Räkna ut RMSE för att se hur stort fel modellen gör i snitt
test_rmse = root_mean_squared_error(y_test, y_pred)

print(f"Slutgiltig RMSE på testsetet: {test_rmse:.2f}")

Slutgiltig RMSE på testsetet: 7059.04
